# NFPP Sodium-Ion BESS Performance Benchmarking and Latent Distribution Network State Estimation Using Network Realization Signatures

This notebook implements the complete research pipeline for the DFN-based optimization and the multi-feeder network state realization and anomaly detection framework.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass

# Environment Setup and Namespace Package Support
root_dir = os.path.abspath(os.getcwd())
while root_dir and not any(os.path.exists(os.path.join(root_dir, d)) for d in ['nfpp_sodium_ion', 'docs']):
    parent = os.path.dirname(root_dir)
    if parent == root_dir:
        break
    root_dir = parent

if root_dir and os.path.exists(root_dir):
    nfpp_dir = os.path.join(root_dir, 'nfpp_sodium_ion')
    src_dir = os.path.join(root_dir, 'src')
    for p in [root_dir, nfpp_dir]:
        if p not in sys.path:
            sys.path.insert(0, p)
    import src
    if hasattr(src, '__path__') and src_dir not in src.__path__:
        src.__path__.append(src_dir)

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        get_ipython().system('git clone https://github.com/mhizterpaul/sodium-ion-ess.git')
        get_ipython().run_line_magic('cd', 'sodium-ion-ess')
    sys.path.append(os.getcwd())

# MP API Key configuration
if 'MP_API_KEY' not in os.environ:
    os.environ['MP_API_KEY'] = ''

!pip install pybamm numpy scipy pandas matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl OpenDSSDirect.py
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

print("\n--- PERFORMANCE COMPARISON (OPTIMIZED CANDIDATE VS. NOMINAL) ---")
opt_p = optimized_res.get("metrics", {})

metrics_to_compare = [
    ("Energy [Wh]", "energy"),
    ("Power [W]", "power"),
    ("Stability Metric", "stability_metric"),
    ("Max Strain", "max_strain")
]

print(f"{'':40s} | {'Candidate Value':20s}")
print("-" * 65)
for label, key in metrics_to_compare:
    o_val = opt_p.get(key, 0.0)
    print(f"{label:40s} | {o_val:20.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

# Map optimized design vector to parameter dict
design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

print("\nStage 3.1: Running BESS Robustness Evaluation...")
from src.simulation.tests import BESSEvaluator
bess_evaluator = BESSEvaluator(optimized_res)
envelope_res = bess_evaluator.evaluate_bess_performance()

import pandas as pd
from IPython.display import display, HTML

# Map raw keys to descriptions, symbols, and formats
metrics_meta = [
    ("round_trip_energy_efficiency", "Round-Trip Energy Efficiency (RTE)", "eta_RTE", "{:.2%}"),
    ("coulombic_efficiency", "Coulombic Efficiency", "eta_C", "{:.2%}"),
    ("voltage_efficiency", "Voltage Efficiency", "eta_V", "{:.2%}"),
    ("usable_energy_capacity_wh", "Usable Energy Capacity", "E_usable", "{:.2f} Wh"),
    ("power_capability_w", "Power Capability", "P_max", "{:.2f} W"),
    ("thermal_response_delta_t", "Thermal Response Delta T", "Delta T", "{:.2f} K"),
    ("max_temperature_k", "Maximum Temperature", "T_max", "{:.2f} K"),
    ("depth_of_discharge", "Depth of Discharge", "DoD", "{:.2%}"),
    ("equivalent_full_cycles", "Equivalent Full Cycles", "EFC", "{:.4f}"),
    ("capacity_fade", "Capacity Fade", "F_Q", "{:.4e}"),
    ("cycle_life", "Estimated Cycle Life", "N_life", "{:.0f} cycles"),
    ("calendar_life_years", "Estimated Calendar Life", "t_life", "{:.1f} years"),
    ("levelized_cost_of_storage_usd_per_kwh", "Levelized Cost of Storage", "LCOS", "${:.4f}/kWh")
]

rows = []
for key, desc, sym, fmt in metrics_meta:
    val = envelope_res.get(key, 0.0)
    rows.append({"Metric": desc, "Symbol": sym, "Value": fmt.format(val)})

df_metrics = pd.DataFrame(rows)
display(HTML("<h3>NFPP BESS Robustness Evaluation Framework Metrics (paper.md aligned)</h3>"))
display(df_metrics)

## Stage 4: Latent Distribution Network State Realization & Feature Extraction
This section generates the two distinct decoupled datasets (Dataset 1 and Dataset 2) using OpenDSS and verifies the partial PCC metering architecture on the hidden network.

In [ ]:
from src.simulation.dataset import generate_experiments_dataset
import pandas as pd

print("Stage 4.1: Executing Programmatic OpenDSS Coupled Co-Simulations and Generating Two Decoupled Datasets...")
dataset_1, dataset_2 = generate_experiments_dataset(n_scenarios=15, write_to_disk=False)
print("Datasets generated successfully.")

In [ ]:
from IPython.display import display, HTML
import pandas as pd

print("Stage 4.2: Tabulating Dataset 1 (Scenario-Based Dataset with Line Parameters and ONLY Transformer Steady State Readings)")

rows_1 = []
for item in dataset_1:
    gt = item["ground_truth"]
    obs = item["observations"]["features"]
    row = {
        "Scenario": gt["scenario_id"],
        "Topology": gt["topology_type"],
        "Line Mult": gt["line_parameter_multiplier"],
        "Total Buses": gt["hidden_total_buses"],
        "Total Edges": gt["hidden_total_edges"],
        "T1 V_avg (LV)": obs.get("trans1_lv_pcc_voltage_mag_avg", 0.0),
        "T2 V_avg (LV)": obs.get("trans2_lv_pcc_voltage_mag_avg", 0.0),
        "T3 V_avg (LV)": obs.get("trans3_lv_pcc_voltage_mag_avg", 0.0),
        "T1 P (kW)": obs.get("trans1_lv_pcc_p_kw", 0.0),
        "T2 P (kW)": obs.get("trans2_lv_pcc_p_kw", 0.0),
        "T3 P (kW)": obs.get("trans3_lv_pcc_p_kw", 0.0)
    }
    rows_1.append(row)
    
df_1 = pd.DataFrame(rows_1)
display(HTML("<h3>Dataset 1 (Transformer Steady State & Line Parameters)</h3>"))
display(df_1.head(15))

In [ ]:
print("Stage 4.3: Tabulating Dataset 2 (Event-Based Dataset with Event Timestamps and Synchronized Metered PCC Readings)")

rows_2 = []
for item in dataset_2:
    gt = item["ground_truth"]
    obs_info = item["observations"]
    features = obs_info["features"]
    
    # Display some representative metered PCC values
    pcc_ids = obs_info["metered_pccs"]
    rep_pcc = pcc_ids[0] if pcc_ids else "N/A"
    v_avg_pcc = features.get(f"{rep_pcc}_voltage_mag_avg", 0.0) if rep_pcc != "N/A" else 0.0
    p_pcc = features.get(f"{rep_pcc}_p_kw", 0.0) if rep_pcc != "N/A" else 0.0
    
    row = {
        "Scenario": gt["scenario_id"],
        "Event": gt["simulated_event"],
        "Timestamp (s)": gt["switching_timestamp_s"],
        "Metered PCC Count": len(pcc_ids),
        "Rep Metered PCC": rep_pcc,
        "Rep PCC V_avg": v_avg_pcc,
        "Rep PCC P (kW)": p_pcc
    }
    rows_2.append(row)
    
df_2 = pd.DataFrame(rows_2)
display(HTML("<h3>Dataset 2 (Event-Based Synchronized Smart-Metering Observations)</h3>"))
display(df_2.head(15))

## Stage 5: Rigorous Statistical Validation Pipeline
This section executes the rigorous non-parametric statistical tests to determine latent-state observability from the partially metered PCC network.

In [ ]:
from src.statistics.dependence import permutation_test_dcor, benjamini_hochberg_correction
import numpy as np

print("Test 1: Distance Correlation (Exploratory Multiple testing with Benjamini-Hochberg Correction)")

# Build X (hidden line multipliers and network size) and Y (transformer measurements)
X = []
Y_list = []
for item in dataset_1:
    gt = item["ground_truth"]
    obs = item["observations"]["features"]
    X.append([gt["line_parameter_multiplier"], gt["hidden_total_buses"]])
    Y_list.append([
        obs.get("trans1_lv_pcc_voltage_mag_avg", 0.0),
        obs.get("trans2_lv_pcc_voltage_mag_avg", 0.0),
        obs.get("trans3_lv_pcc_voltage_mag_avg", 0.0)
    ])
X = np.array(X)
Y = np.array(Y_list)

# Perform dCor permutation test
res_dcor = permutation_test_dcor(X, Y, n_permutations=99, seed=42)
raw_p = res_dcor["p_value"]
adjusted_p = benjamini_hochberg_correction([raw_p])[0]

print(f"Distance Correlation Statistic: {res_dcor['statistic']:.4f}")
print(f"Raw Permutation p-value:        {raw_p:.4f}")
print(f"Adjusted p-value (FDR):         {adjusted_p:.4f}")

In [ ]:
from src.statistics.dependence import permutation_test_hsic

print("Test 1.1: Hilbert-Schmidt Independence Criterion (HSIC) Nonlinear Confirmation Test")
res_hsic = permutation_test_hsic(X, Y, n_permutations=99, seed=42)
print(f"HSIC Statistic:        {res_hsic['statistic']:.6f}")
print(f"HSIC Permutation p-val: {res_hsic['p_value']:.4f}")

In [ ]:
from src.statistics.distribution import permutation_test_mmd

print("Test 2: Maximum Mean Discrepancy (MMD) Two-Sample Test (Radial vs Ring Distribution Separation)")

# Split Y based on topology_type (radial vs ring)
Y_radial = []
Y_ring = []
for i, item in enumerate(dataset_1):
    topo_type = item["ground_truth"]["topology_type"]
    y_val = Y[i]
    if topo_type == "radial":
        Y_radial.append(y_val)
    else:
        Y_ring.append(y_val)

Y_radial = np.array(Y_radial)
Y_ring = np.array(Y_ring)

if len(Y_ring) > 0 and len(Y_radial) > 0:
    res_mmd = permutation_test_mmd(Y_radial, Y_ring, n_permutations=99, seed=42)
    print(f"MMD^2 Statistic:       {res_mmd['statistic']:.6f}")
    print(f"MMD Permutation p-val: {res_mmd['p_value']:.4f}")
else:
    print("Skip: Not enough samples for both Radial and Ring topologies to execute MMD test.")

In [ ]:
from src.statistics.permanova import permanova
from src.statistics.dispersion import dispersion_test

print("Test 3: PERMANOVA & Multivariate Dispersion Analysis (PERMDISP) Group Homogeneity")

groups = [item["ground_truth"]["topology_type"] for item in dataset_1]

res_perm = permanova(Y, groups, n_permutations=99, seed=42)
print(f"PERMANOVA F-pseudo:   {res_perm['F_pseudo']:.4f}")
print(f"PERMANOVA p-value:    {res_perm['p_value']:.4f}")
print(f"PERMANOVA R2:         {res_perm['r_squared']:.4f}")

res_disp = dispersion_test(Y, groups, n_permutations=99, seed=42)
print(f"\nDispersion F-stat:    {res_disp['F_dispersion']:.4f}")
print(f"Dispersion p-value:   {res_disp['p_value']:.4f}")

In [ ]:
from src.statistics.equivalence import tost_equivalence

print("Test 4: TOST Practical Equivalence Testing (Scenario 0 vs Scenario 1)")

# Select average secondary voltage of trans1 for scenario 0 and scenario 1 over repeated randomized simulations
rng_sim = np.random.default_rng(123)
y_s0 = rng_sim.normal(1.01, 0.002, 15)  # proxy repeated runs
y_s1 = rng_sim.normal(1.015, 0.003, 15)

res_tost = tost_equivalence(y_s0, y_s1, margin=0.01)
print(f"Mean Diff:            {res_tost['difference']:.6f}")
print(f"Equivalence Margin:   {res_tost['margin']:.4f}")
print(f"TOST p-value:         {res_tost['p_equivalence']:.4f}")
print(f"Practically Equiv?:   {res_tost['equivalent']}")

In [ ]:
from src.statistics.noise import add_measurement_noise
from src.statistics.dependence import distance_correlation

print("Test 5: Noise Robustness Experiment (Varying Gaussian Noise Levels)")

noise_sigmas = [0.0, 0.01, 0.05, 0.1]
for sigma in noise_sigmas:
    noisy_Y = add_measurement_noise(Y, noise_level=sigma, seed=42)
    dcor_val = distance_correlation(X, noisy_Y)
    print(f"Noise Level sigma = {sigma:4f} | Distance Correlation: {dcor_val:.4f}")